In [0]:
%sql
USE CATALOG crp_arb_dev;

DROP TABLE silver.crypto_data_15m;
DROP TABLE silver.crypto_data_24h;

In [0]:
from pyspark.sql import functions as F

bronze_stream = spark.readStream \
    .option("startingOffsets", "earliest") \
    .table("crp_arb_dev.bronze.crypto_data_delta")

# need to be checked
base_silver_df = bronze_stream.select(
    F.col("symbol").alias("ticker"),
    F.col("interval"),
    F.col("open").cast("double"),
    F.col("close").cast("double"),
    F.col("kline")["h"].cast("double").alias("high_price"),
    F.col("kline")["l"].cast("double").alias("low_price"),
    F.col("kline")["v"].cast("double").alias("trading_volume"),
    F.col("ingested_at").alias("event_timestamp")
)

container_path = "abfss://crp-arb-cnt@sadlsdev.dfs.core.windows.net"

silver_15m_query = base_silver_df\
    .filter(F.lower(F.col("interval")) == "15m")\
    .writeStream\
    .format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", f"{container_path}/_checkpoints/silver_15m") \
    .trigger(availableNow=True) \
    .toTable("silver.crypto_data_15m")

silver_24h_query = base_silver_df\
    .filter(F.col("interval") == "1d")\
    .writeStream\
    .format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", f"{container_path}/_checkpoints/silver_1d") \
    .trigger(availableNow=True) \
    .toTable("silver.crypto_data_24h")

silver_15m_query.awaitTermination()
silver_24h_query.awaitTermination()